<a href="https://colab.research.google.com/github/khiemp110/template/blob/main/comfy.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ComfyUI — Colab launcher có lưu Drive và đăng nhập

Chọn **Runtime → Change runtime type → GPU**, dùng Python **3.12+**, rồi chọn **Runtime → Run all** để chạy các cell từ trên xuống.
Notebook kiểm tra GPU thực tế; lựa chọn T4 trong metadata không đảm bảo Colab cấp T4.

1. Bộ tải model nền đã được nhúng sẵn trong notebook và tự bật khi mở phiên.
   Không cần thêm cell hoặc tải file worker riêng. Bạn có thể dùng model đã có tại
   `MyDrive/ComfyUI/models/checkpoints`, hoặc yêu cầu agent tải model qua script Python
   sau khi phiên đã chạy. Notebook không tự tải model khi chưa có yêu cầu.
2. Tạo tài khoản [ngrok](https://dashboard.ngrok.com/get-started/your-authtoken).
   Trong **Secrets** (biểu tượng chìa khóa của Colab), thêm `NGROK_AUTHTOKEN` và
   `COMFYUI_PASSWORD` (ít nhất 6 ký tự ASCII, không có khoảng trắng), bật quyền truy cập notebook.
   Nếu chưa có Secrets, notebook hỏi bằng ô nhập ẩn. Không dán bí mật vào code hay output.
3. Cell cuối giữ phiên chạy. Mở link HTTPS và đăng nhập bằng tên `comfyui` cùng mật khẩu đã đặt.
   Notebook kiểm tra cả truy cập bị chặn khi chưa đăng nhập và truy cập thành công khi đăng nhập.
   Nếu ngrok không chấp nhận token/tính năng xác thực của tài khoản, phiên sẽ dừng, không bỏ mật khẩu.
4. Bấm **Stop** ở cell cuối để dừng server và tunnel. Mặc định tự dừng sau **4 giờ**;
   đổi `MAX_SESSION_HOURS` trước khi chạy nếu cần. Sau đó ngắt/xóa runtime trong menu Colab
   để kết thúc việc sử dụng máy GPU; dừng server không tự giải phóng runtime.

Ảnh dùng node **Save Image** được lưu tại `MyDrive/ComfyUI/output`; input và dữ liệu người dùng
(gồm workflow đã lưu trong giao diện) cũng nằm trên Drive. Ảnh chỉ xem trước vẫn là dữ liệu tạm.
Checkpoint, LoRA, VAE và các loại model thông dụng được đọc trực tiếp từ Drive, không xóa thư mục cũ.
Đọc model từ Drive có thể chậm; notebook không tự sao chép model nhiều GB sang ổ cục bộ.

**Custom nodes:** mặc định không chạy custom nodes. Muốn dùng mã bạn tin tưởng từ
`MyDrive/ComfyUI/custom_nodes`, bật `ENABLE_CUSTOM_NODES`; dependencies riêng của node cần cài riêng.
Mã chạy trong runtime có thể đọc Drive đã mount.

**Lỗi Drive:** nếu thấy `credential propagation was unsuccessful`, hoàn tất đăng nhập/cấp quyền Google
trong Colab; nếu vẫn lỗi, dùng **Runtime → Disconnect and delete runtime**, kết nối lại GPU và chạy lại.
Không có cách sửa code nào thay thế bước Google cấp quyền. Không tự retry mount liên tục.

**Dùng lại kết nối Drive:** notebook bỏ qua `drive.mount()` nếu Drive vẫn được mount và
`MyDrive` truy cập được trong runtime hiện tại; giữ `FORCE_REMOUNT_DRIVE = False`.
Nếu chỉ chạy lại ComfyUI sau khi bấm Stop, chạy lại cell cuối. Khi runtime bị xóa/thu hồi,
Google có thể yêu cầu cấp quyền lại. Dùng **File → Save a copy in Drive** và mở lại cùng bản
cá nhân đó; Colab có thể tự mount ở phiên sau nếu notebook đạt các kiểm tra của Google,
nhưng notebook không thể đảm bảo bỏ xác thực ở mọi runtime mới.

ComfyUI được ghim ở một commit, SDK ngrok ở một phiên bản; các dependencies còn lại theo requirements
của commit đó và bộ PyTorch tương thích đang có trên runtime. Để nâng cấp, chủ động đổi cấu hình.
Xem [Colab FAQ](https://research.google.com/colaboratory/faq.html),
[ngrok SDK](https://ngrok.github.io/ngrok-python/) và
[cấu hình model ComfyUI](https://github.com/Comfy-Org/ComfyUI/blob/master/extra_model_paths.yaml.example).

**Tải model nền:** launcher khởi động bộ tải nền cùng phiên ComfyUI. Agent gửi manifest
qua API userdata đã có Basic Auth; bộ tải chỉ nhận `.safetensors` từ Hugging Face,
ghim revision và xác minh SHA256. Model được lưu trên Drive. Không cần dừng ComfyUI
để tải thêm checkpoint/LoRA/VAE. Xem `COMFY_AGENT_GUIDE.md` và `install-model` của script.


In [ ]:
# 1. Cấu hình — chạy lại từ cell này sau khi đổi giá trị.
import asyncio
import base64
import getpass
import json
import os
from pathlib import Path
import re
import signal
import socket
import subprocess
import sys
import time
import urllib.error
import urllib.request

try:
    from google.colab import drive, userdata
except ImportError:
    raise RuntimeError("Hãy mở notebook bằng Google Colab với GPU runtime.") from None
if sys.version_info < (3, 12):
    raise RuntimeError("Launcher này cần Python 3.12+ trên Google Colab.")

COMFYUI_REF = "0228a9d73cd7c2daf8f61fe13d56b12efe307171"
NGROK_VERSION = "1.7.0"
DRIVE_ROOT = Path("/content/drive/MyDrive/ComfyUI")
COMFY_DIR = Path("/content/ComfyUI")
WORK_DIR = Path("/content/comfyui-launcher")
PORT = 3309
STARTUP_TIMEOUT_SECONDS = 300
INSTALL_TIMEOUT_SECONDS = 1800
MAX_SESSION_HOURS = 4
FORCE_REMOUNT_DRIVE = False
ENABLE_CUSTOM_NODES = False
LOW_VRAM = False  # Chỉ bật nếu workflow báo thiếu VRAM; có thể làm chậm.
LOGIN_USERNAME = "comfyui"

if not re.fullmatch(r"[0-9a-f]{40}", COMFYUI_REF):
    raise ValueError("COMFYUI_REF phải là commit SHA đầy đủ (40 ký tự).")
if not 1024 <= PORT <= 65535:
    raise ValueError("PORT phải nằm trong khoảng 1024..65535.")
if any(value <= 0 for value in (
    STARTUP_TIMEOUT_SECONDS, INSTALL_TIMEOUT_SECONDS, MAX_SESSION_HOURS
)):
    raise ValueError("Timeout và số giờ chạy phải lớn hơn 0.")
if not re.fullmatch(r"[A-Za-z0-9_-]{1,64}", LOGIN_USERNAME):
    raise ValueError("Tên đăng nhập chỉ dùng chữ ASCII, số, dấu _ hoặc -.")
DRIVE_READY = SETUP_READY = STORAGE_READY = False
print("Cấu hình sẵn sàng. Không có token/mật khẩu nào được lưu vào notebook.")

In [ ]:
# 2. Kiểm tra GPU trước khi cài; mount Drive một lần, báo lỗi có hướng xử lý.
DRIVE_READY = SETUP_READY = STORAGE_READY = False

def run_command(arguments, *, cwd=None, capture=False, timeout=180):
    return subprocess.run(
        [str(arg) for arg in arguments], cwd=cwd, check=True, text=True,
        stdout=subprocess.PIPE if capture else None,
        stderr=subprocess.PIPE if capture else None, timeout=timeout,
    )

def probe_torch():
    # Probe trong tiến trình riêng: không giữ CUDA/torch trong kernel notebook.
    script = """
import importlib.metadata as metadata
import json
import torch
import torchvision
if not torch.cuda.is_available():
    raise RuntimeError('PyTorch không truy cập được GPU CUDA.')
sample = torch.ones(1, device='cuda')
torch.cuda.synchronize()
packages = {}
for name in ('torch', 'torchvision', 'torchaudio'):
    try:
        packages[name] = metadata.version(name)
    except metadata.PackageNotFoundError:
        pass
print(json.dumps({'packages': packages, 'gpu': torch.cuda.get_device_name(0),
                 'cuda': torch.version.cuda}))
"""
    try:
        result = run_command([sys.executable, "-c", script], capture=True)
        return json.loads(result.stdout.strip().splitlines()[-1])
    except (subprocess.SubprocessError, ValueError, IndexError):
        raise RuntimeError(
            "GPU/PyTorch/torchvision chưa hoạt động. Chọn GPU runtime mới trong Colab "
            "rồi chạy lại. Không tự ép cài lại CUDA/PyTorch lên môi trường đang lỗi."
        ) from None

gpu_info = probe_torch()
print(f"GPU thực tế: {gpu_info['gpu']} | CUDA của PyTorch: {gpu_info['cuda']}")
try:
    mount_path = Path("/content/drive")
    # Colab có thể yêu cầu xác thực ngay khi gọi mount(), kể cả trước khi kiểm tra mount cũ.
    if (not FORCE_REMOUNT_DRIVE and mount_path.is_mount()
            and (mount_path / "MyDrive").is_dir()):
        print("Dùng lại kết nối Drive hiện tại; không yêu cầu mount lại.")
    else:
        drive.mount(str(mount_path), force_remount=FORCE_REMOUNT_DRIVE)
except Exception:
    raise RuntimeError(
        "Không mount được Drive. Hoàn tất đăng nhập/cấp quyền Google trong Colab. "
        "Nếu gặp 'credential propagation was unsuccessful', hãy Disconnect and delete "
        "runtime, kết nối lại GPU rồi chạy lại từ đầu."
    ) from None
if not Path("/content/drive/MyDrive").is_dir():
    raise RuntimeError("Drive chưa sẵn sàng: không tìm thấy /content/drive/MyDrive.")
DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
WORK_DIR.mkdir(parents=True, exist_ok=True)
DRIVE_READY = True
print(f"Drive sẵn sàng: {DRIVE_ROOT}")

In [ ]:
# 3. Checkout commit cố định, giữ nguyên PyTorch đang hoạt động.
SETUP_READY = STORAGE_READY = False
if not DRIVE_READY:
    raise RuntimeError("Chạy thành công cell GPU và Drive trước.")

def ensure_comfy_checkout():
    repository = "https://github.com/Comfy-Org/ComfyUI.git"
    if not (COMFY_DIR / ".git").exists():
        if COMFY_DIR.exists() and any(COMFY_DIR.iterdir()):
            raise RuntimeError(f"{COMFY_DIR} có dữ liệu nhưng không phải Git repo; không ghi đè.")
        COMFY_DIR.mkdir(parents=True, exist_ok=True)
        run_command(["git", "init", COMFY_DIR])
        run_command(["git", "remote", "add", "origin", repository], cwd=COMFY_DIR)
    origin = run_command(
        ["git", "remote", "get-url", "origin"], cwd=COMFY_DIR, capture=True
    ).stdout.strip()
    allowed = {repository, "https://github.com/comfyanonymous/ComfyUI.git"}
    if origin.rstrip("/").removesuffix(".git") not in {
        url.removesuffix(".git") for url in allowed
    }:
        raise RuntimeError("Repo hiện tại có origin khác ComfyUI; không tự thay thế.")
    dirty = run_command(
        ["git", "status", "--porcelain", "--untracked-files=no"],
        cwd=COMFY_DIR, capture=True,
    ).stdout.strip()
    if dirty:
        raise RuntimeError("ComfyUI có file tracked đã chỉnh sửa; lưu thay đổi trước khi checkout.")
    current = subprocess.run(
        ["git", "rev-parse", "--verify", "HEAD"], cwd=COMFY_DIR,
        text=True, capture_output=True, timeout=30,
    ).stdout.strip()
    if current != COMFYUI_REF:
        run_command(["git", "fetch", "--depth", "1", "origin", COMFYUI_REF], cwd=COMFY_DIR)
        run_command(["git", "checkout", "--detach", COMFYUI_REF], cwd=COMFY_DIR)
    print(f"ComfyUI commit: {COMFYUI_REF}")

ensure_comfy_checkout()
gpu_info = probe_torch()
constraints = WORK_DIR / "torch-constraints.txt"
constraints.write_text(
    "".join(f"{name}=={version}\n" for name, version in gpu_info["packages"].items()),
    encoding="utf-8",
)
run_command([
    sys.executable, "-m", "pip", "install", "--disable-pip-version-check", "--no-input",
    "-r", COMFY_DIR / "requirements.txt", "-c", constraints, f"ngrok=={NGROK_VERSION}",
], timeout=INSTALL_TIMEOUT_SECONDS)
verified_gpu = probe_torch()
if verified_gpu["packages"] != gpu_info["packages"]:
    raise RuntimeError("Bộ PyTorch đã thay đổi ngoài dự kiến; tạo GPU runtime mới rồi chạy lại.")
SETUP_READY = True
print("Dependencies sẵn sàng; bộ PyTorch/CUDA hiện có đã được giữ lại và kiểm tra.")

In [ ]:
# 4. Dùng cấu hình model và tham số đường dẫn, không xóa/symlink thư mục.
STORAGE_READY = False
if not SETUP_READY or not DRIVE_READY:
    raise RuntimeError("Chạy thành công các cell trước.")
import yaml

MODEL_PATHS = {
    "checkpoints": "models/checkpoints", "loras": "models/loras",
    "vae": "models/vae", "text_encoders": "models/text_encoders\nmodels/clip",
    "diffusion_models": "models/diffusion_models\nmodels/unet",
    "clip_vision": "models/clip_vision", "embeddings": "models/embeddings",
    "controlnet": "models/controlnet\nmodels/t2i_adapter",
    "upscale_models": "models/upscale_models", "vae_approx": "models/vae_approx",
    "configs": "models/configs", "style_models": "models/style_models",
}
if ENABLE_CUSTOM_NODES:
    MODEL_PATHS["custom_nodes"] = "custom_nodes"
for paths in MODEL_PATHS.values():
    for relative in paths.splitlines():
        (DRIVE_ROOT / relative).mkdir(parents=True, exist_ok=True)
for name in ("output", "input", "user"):
    directory = DRIVE_ROOT / name
    directory.mkdir(parents=True, exist_ok=True)
    # Test actual write access; a mounted Drive can still be full/read-only.
    import tempfile
    with tempfile.TemporaryFile(dir=directory) as probe:
        probe.write(b"drive-write-check")
        probe.flush()

MODEL_CONFIG = WORK_DIR / "extra_model_paths.yaml"
MODEL_CONFIG.write_text(yaml.safe_dump({"google_drive": {
    "base_path": str(DRIVE_ROOT), "is_default": True, **MODEL_PATHS,
}}, sort_keys=False), encoding="utf-8")
checkpoint_dir = DRIVE_ROOT / "models/checkpoints"
if not any(p.suffix.lower() in {".safetensors", ".ckpt", ".pt", ".bin"}
           for p in checkpoint_dir.rglob("*") if p.is_file()):
    print(f"Chưa thấy checkpoint trong {checkpoint_dir}. Thêm model nếu workflow cần checkpoint.")
STORAGE_READY = True
print(f"Ảnh Save Image → {DRIVE_ROOT / 'output'}")
print(f"Input → {DRIVE_ROOT / 'input'} | Dữ liệu/workflow → {DRIVE_ROOT / 'user'}")

In [ ]:
# 4b. Bộ tải model nền — agent gửi job qua userdata, không cần dừng ComfyUI.
"""Colab background downloader; requests/status travel via authenticated ComfyUI userdata."""
import builtins
import hashlib
import json
import os
from pathlib import Path
import re
import shutil
import subprocess
import tempfile
import threading
import time
import urllib.parse

MODEL_DOWNLOAD_NAMESPACE = '_agent_models'
MODEL_DOWNLOAD_FOLDERS = {
    'checkpoints', 'loras', 'vae', 'text_encoders', 'diffusion_models',
    'clip_vision', 'embeddings', 'controlnet', 'upscale_models', 'vae_approx',
}


def validate_model_request(job):
    if not isinstance(job, dict):
        raise ValueError('Job must be a JSON object.')
    for key, pattern in (
        ('id', r'[0-9a-f]{32}'),
        ('repo_id', r'[A-Za-z0-9_.-]+/[A-Za-z0-9_.-]+'),
        ('revision', r'[0-9a-f]{40}'),
        ('sha256', r'[0-9a-f]{64}'),
        ('filename', r'[A-Za-z0-9][A-Za-z0-9_.-]*\.safetensors'),
    ):
        if not isinstance(job.get(key), str) or not re.fullmatch(pattern, job[key]):
            raise ValueError(f'Invalid {key}.')
    source = job.get('source_filename')
    if (not isinstance(source, str) or not source.endswith('.safetensors')
            or any(not re.fullmatch(r'[A-Za-z0-9][A-Za-z0-9_.-]*', part)
                   for part in source.split('/'))):
        raise ValueError('Invalid source_filename.')
    if job.get('folder') not in MODEL_DOWNLOAD_FOLDERS:
        raise ValueError('Invalid destination model folder.')
    if type(job.get('size')) is not int or not 0 < job['size'] <= 100_000_000_000:
        raise ValueError('Invalid expected size (maximum 100 GB).')
    return job


class ModelDownloadWorker(threading.Thread):
    def __init__(self, drive_root):
        super().__init__(name='comfy-model-downloader', daemon=True)
        self.drive_root = Path(drive_root).resolve()
        self.root = self.drive_root / 'user' / 'default' / MODEL_DOWNLOAD_NAMESPACE
        self.requests = self.root / 'requests'
        self.statuses = self.root / 'status'
        self.models = self.drive_root / 'models'
        self.partials = self.models / '.agent-downloads'
        self.stop_event = threading.Event()
        for directory in (self.requests, self.statuses, self.partials):
            self.confined(directory).mkdir(parents=True, exist_ok=True)
        self.last_heartbeat = 0

    def confined(self, path):
        path = Path(path)
        if not path.resolve().is_relative_to(self.drive_root):
            raise ValueError('Destination escapes the configured Drive root.')
        return path

    def write_json(self, path, value):
        path = self.confined(path)
        fd, temporary = tempfile.mkstemp(prefix='.write-', dir=path.parent)
        try:
            with os.fdopen(fd, 'w', encoding='utf-8') as stream:
                json.dump(value, stream, ensure_ascii=False)
            os.replace(temporary, path)
        finally:
            if os.path.exists(temporary):
                os.unlink(temporary)

    def heartbeat(self, active_job=None, force=False):
        now = time.time()
        if force or now - self.last_heartbeat >= 15:
            self.write_json(self.root / 'worker.json', {
                'protocol': 1, 'state': 'ready', 'heartbeat_at': now,
                'active_job': active_job, 'folders': sorted(MODEL_DOWNLOAD_FOLDERS),
            })
            self.last_heartbeat = now

    def status(self, job, state, **extra):
        self.write_json(self.statuses / (job['id'] + '.json'), {
            'job_id': job['id'], 'state': state, 'filename': job.get('filename'),
            'expected_bytes': job.get('size'), 'updated_at': time.time(), **extra,
        })
        self.heartbeat(job['id'])

    def verify(self, path, job):
        if path.stat().st_size != job['size']:
            raise ValueError('Downloaded file size does not match the manifest.')
        digest = hashlib.sha256()
        last_update = 0
        with path.open('rb') as stream:
            while chunk := stream.read(8 * 1024 * 1024):
                digest.update(chunk)
                if time.monotonic() - last_update > 10:
                    self.status(job, 'verifying', downloaded_bytes=job['size'])
                    last_update = time.monotonic()
        if digest.hexdigest() != job['sha256']:
            raise ValueError('SHA256 mismatch; model was not installed.')

    def install(self, job):
        target = self.confined(self.models / job['folder'] / job['filename'])
        target.parent.mkdir(parents=True, exist_ok=True)
        if target.exists():
            self.verify(target, job)
            self.status(job, 'completed', path=str(target), downloaded_bytes=job['size'], reused=True)
            return
        partial = self.confined(self.partials / (job['id'] + '.part'))
        already = partial.stat().st_size if partial.exists() else 0
        if already > job['size']:
            raise ValueError('Partial file is larger than expected.')
        if shutil.disk_usage(target.parent).free < job['size'] - already:
            raise ValueError('Insufficient free space for this model.')
        if shutil.which('curl') is None:
            raise ValueError('Colab runtime has no curl executable.')
        repo = urllib.parse.quote(job['repo_id'], safe='/')
        source = urllib.parse.quote(job['source_filename'], safe='/')
        # Only pinned Hugging Face HTTPS sources; clients cannot supply arbitrary URLs.
        url = f"https://huggingface.co/{repo}/resolve/{job['revision']}/{source}?download=true"
        self.status(job, 'downloading', downloaded_bytes=already)
        if already < job['size']:
            log_path = self.confined(self.partials / (job['id'] + '.log'))
            with log_path.open('ab') as log:
                process = subprocess.Popen([
                    'curl', '--fail', '--location', '--silent', '--show-error',
                    '--retry', '3', '--retry-delay', '3', '--connect-timeout', '30',
                    '--max-time', '3600', '--continue-at', '-', '--output', str(partial), url,
                ], stdout=subprocess.DEVNULL, stderr=log)
                try:
                    while process.poll() is None:
                        if self.stop_event.wait(5):
                            raise RuntimeError('Model downloader was stopped.')
                        downloaded = partial.stat().st_size if partial.exists() else 0
                        if downloaded > job['size']:
                            raise ValueError('Download exceeded the expected file size.')
                        self.status(job, 'downloading', downloaded_bytes=downloaded)
                    if process.returncode:
                        raise RuntimeError(f'curl failed with exit code {process.returncode}; partial file retained.')
                finally:
                    if process.poll() is None:
                        process.terminate()
                        try:
                            process.wait(timeout=5)
                        except subprocess.TimeoutExpired:
                            process.kill()
                            process.wait(timeout=5)
        self.verify(partial, job)
        if target.exists():
            raise ValueError('Another file appeared at the destination; refusing to overwrite it.')
        partial.rename(target)
        self.status(job, 'completed', path=str(target), downloaded_bytes=job['size'], sha256=job['sha256'])

    def run(self):
        while not self.stop_event.is_set():
            try:
                self.heartbeat()
                for request_path in sorted(self.requests.glob('*.json')):
                    if not re.fullmatch(r'[0-9a-f]{32}\.json', request_path.name):
                        continue
                    status_path = self.confined(self.statuses / request_path.name)
                    if status_path.exists():
                        old = json.loads(status_path.read_text(encoding='utf-8'))
                        if old.get('state') in ('completed', 'failed'):
                            continue
                    job = {'id': request_path.stem}
                    try:
                        self.confined(request_path)
                        if request_path.stat().st_size > 16384:
                            raise ValueError('Job manifest is too large.')
                        loaded = json.loads(request_path.read_text(encoding='utf-8'))
                        validate_model_request(loaded)
                        if loaded['id'] != request_path.stem:
                            raise ValueError('Job ID does not match request filename.')
                        job = loaded
                        self.install(job)
                    except Exception as error:
                        self.status(job, 'failed', error=str(error))
                    if self.stop_event.is_set():
                        return
                self.heartbeat()
            except Exception as error:
                # A temporary Drive I/O error must not crash the ComfyUI session.
                print(f'Model downloader: {type(error).__name__}; retrying Drive access.', flush=True)
            self.stop_event.wait(3)


def start_model_download_worker(drive_root='/content/drive/MyDrive/ComfyUI'):
    root = Path(drive_root).resolve()
    if not root.is_dir():
        raise RuntimeError('Mount Drive và chạy cell chuẩn bị thư mục trước.')
    previous = getattr(builtins, '_comfy_model_download_worker', None)
    if previous is not None and previous.is_alive():
        if previous.drive_root != root:
            raise RuntimeError('Một model downloader đang chạy với Drive root khác.')
        return previous
    worker = ModelDownloadWorker(root)
    worker.heartbeat(force=True)
    worker.start()
    builtins._comfy_model_download_worker = worker
    print('MODEL_DOWNLOADER_READY: agent có thể gửi yêu cầu tải model khi ComfyUI đang chạy.', flush=True)
    return worker


In [ ]:
# 5. Hàm khởi động, xác thực, timeout và cleanup. Chưa mở tunnel ở cell này.
def read_secret(name, *, validator=None, requirement="không được để trống"):
    try:
        value = userdata.get(name)
    except Exception:
        value = None
    while True:
        if value and (validator is None or validator(value)):
            return value
        if value is not None:
            print(f"{name} chưa hợp lệ: {requirement}. Nhập lại; bấm Stop để hủy.")
        value = getpass.getpass(f"Nhập {name} ({requirement}, ẩn): ")


def valid_comfy_password(value):
    return (len(value) >= 6 and value.isascii()
            and all(ch.isprintable() and not ch.isspace() for ch in value))

def port_is_open(port):
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as sock:
        sock.settimeout(1)
        return sock.connect_ex(("127.0.0.1", port)) == 0

def fetch_stats(url, credentials=None):
    # Header bypasses only ngrok's browser warning, never Basic Auth.
    headers = {"ngrok-skip-browser-warning": "true"}
    if credentials is not None:
        encoded = base64.b64encode(credentials.encode("ascii")).decode("ascii")
        headers["Authorization"] = f"Basic {encoded}"
    request = urllib.request.Request(url.rstrip("/") + "/system_stats", headers=headers)
    try:
        with urllib.request.urlopen(request, timeout=5) as response:
            stats = json.load(response)
            if response.status != 200 or not isinstance(stats, dict) or "system" not in stats:
                raise RuntimeError("Dịch vụ chưa trả về system_stats hợp lệ của ComfyUI.")
            return stats
    except urllib.error.HTTPError as error:
        # HTTPError also owns a response/socket, including on failed authenticated requests.
        error.close()
        raise

def verify_public_auth(url, credentials):
    try:
        fetch_stats(url)
    except urllib.error.HTTPError as error:
        try:
            if error.code != 401 or not error.headers.get("WWW-Authenticate", "").lower().startswith("basic"):
                raise RuntimeError("Endpoint chưa xác nhận được bảo vệ bằng Basic Auth.") from None
        finally:
            error.close()
    else:
        raise RuntimeError("Endpoint cho truy cập không đăng nhập; đóng phiên ngay.")
    fetch_stats(url, credentials)

async def wait_until_ready(process, url):
    deadline = time.monotonic() + STARTUP_TIMEOUT_SECONDS
    while time.monotonic() < deadline:
        if process.poll() is not None:
            raise RuntimeError(f"ComfyUI thoát sớm (exit code {process.returncode}).")
        try:
            await asyncio.to_thread(fetch_stats, url)
            return
        except (urllib.error.URLError, OSError, ValueError, RuntimeError):
            await asyncio.sleep(1)
    raise TimeoutError(f"ComfyUI chưa sẵn sàng sau {STARTUP_TIMEOUT_SECONDS} giây.")

def stop_process(process):
    # Only the process group created by this launcher; never pkill unrelated Python.
    try:
        os.killpg(process.pid, signal.SIGTERM)
    except ProcessLookupError:
        pass
    try:
        process.wait(timeout=10)
    except subprocess.TimeoutExpired:
        pass
    # Also stop descendants if the main process exited before them.
    try:
        os.killpg(process.pid, signal.SIGKILL)
    except ProcessLookupError:
        pass
    process.wait(timeout=5)

def print_log_tail(path, line_count=50):
    from collections import deque
    if path.exists():
        with path.open(encoding="utf-8", errors="replace") as stream:
            print("".join(deque(stream, maxlen=line_count)))

async def launch_comfyui():
    if not (DRIVE_READY and SETUP_READY and STORAGE_READY):
        raise RuntimeError("Chạy thành công tất cả cell phía trên trước.")
    start_model_download_worker(DRIVE_ROOT)
    if port_is_open(PORT):
        raise RuntimeError(f"Cổng {PORT} đang được dùng. Dừng phiên cũ trước; không tự kill tiến trình lạ.")
    import ngrok

    token = read_secret(
        "NGROK_AUTHTOKEN", validator=lambda value: bool(value.strip()),
        requirement="dán Authtoken của ngrok, không để trống",
    ).strip()
    password = read_secret(
        "COMFYUI_PASSWORD", validator=valid_comfy_password,
        requirement="ít nhất 6 ký tự không dấu, không khoảng trắng",
    )
    credentials = f"{LOGIN_USERNAME}:{password}"
    command = [
        sys.executable, "-u", "main.py", "--listen", "127.0.0.1", "--port", str(PORT),
        "--disable-auto-launch", "--extra-model-paths-config", str(MODEL_CONFIG),
        "--output-directory", str(DRIVE_ROOT / "output"),
        "--input-directory", str(DRIVE_ROOT / "input"),
        "--user-directory", str(DRIVE_ROOT / "user"),
    ]
    if not ENABLE_CUSTOM_NODES:
        command.append("--disable-all-custom-nodes")
    if LOW_VRAM:
        command.append("--lowvram")
    log_path = WORK_DIR / "comfyui.log"
    process = listener = None
    try:
        with log_path.open("w", encoding="utf-8") as log:
            process = subprocess.Popen(
                command, cwd=COMFY_DIR, stdout=log, stderr=subprocess.STDOUT,
                start_new_session=True,
            )
            print(f"Đang khởi động ComfyUI. Log: {log_path}")
            await wait_until_ready(process, f"http://127.0.0.1:{PORT}")
            try:
                listener = await asyncio.wait_for(ngrok.forward(
                    f"127.0.0.1:{PORT}", authtoken=token,
                    basic_auth=[credentials], schemes=["HTTPS"],
                ), timeout=60)
            except Exception:
                raise RuntimeError(
                    "Không mở được ngrok HTTPS có xác thực. Kiểm tra token, kết nối và "
                    "quyền Basic Auth của tài khoản tại ngrok dashboard; không mở tunnel không mật khẩu."
                ) from None
            url = listener.url()
            if not url.startswith("https://"):
                raise RuntimeError("Tunnel không dùng HTTPS; đóng phiên.")
            # Giữ nguyên lỗi gốc để phân biệt HTTP, timeout và lỗi xác thực.
            await asyncio.to_thread(verify_public_auth, url, credentials)
            token = password = credentials = None
            print(f"ComfyUI: {url}")
            print(f"Đăng nhập: {LOGIN_USERNAME} + COMFYUI_PASSWORD. Mật khẩu không được in ra.")
            print(f"Tự dừng sau {MAX_SESSION_HOURS} giờ; bấm Stop ở cell này để dừng sớm.")
            deadline = time.monotonic() + MAX_SESSION_HOURS * 3600
            while time.monotonic() < deadline:
                if process.poll() is not None:
                    raise RuntimeError(f"ComfyUI đã thoát (exit code {process.returncode}).")
                await asyncio.sleep(1)
            print("Đã hết thời gian phiên; đang dừng.")
    except (KeyboardInterrupt, asyncio.CancelledError):
        print("Đã nhận lệnh dừng.")
        return  # Stop là kết thúc bình thường; finally vẫn cleanup.
    except Exception:
        print_log_tail(log_path)
        raise
    finally:
        token = password = credentials = None
        # Stop backend first: even a failed tunnel close cannot leave a live ComfyUI exposed.
        try:
            if process is not None:
                stop_process(process)
        finally:
            if listener is not None:
                try:
                    await asyncio.wait_for(listener.close(), timeout=10)
                except Exception:
                    print("Không xác nhận được tunnel đã đóng; backend đã nhận lệnh dừng.")
            print("Cleanup hoàn tất. Ngắt/xóa runtime trong menu Colab khi dùng xong GPU.")

In [ ]:
# 6. Chạy phiên — giữ cell hoạt động, bấm Stop để kết thúc và cleanup.
await launch_comfyui()